# Fine-Tuning a Transformer Model with QLoRA on Azure Databricks

## Overview
This notebook demonstrates parameter-efficient fine-tuning using QLoRA (Quantized Low-Rank Adaptation) on Azure Databricks.

QLoRA combines LoRA with 4-bit quantization to enable:
- Efficient fine-tuning of large models on limited hardware
- Reduced memory footprint (up to 4x smaller than LoRA)
- Maintained performance while using significantly less GPU memory

## Objectives
- Setup the environment with PEFT and bitsandbytes libraries
- Load datasets
- Configure QLoRA parameters with 4-bit quantization
- Train the model with QLoRA adapters
- Evaluate model performance and save results

## Author
- Name: Alessandro Armillotta
- Date: 01/07/2026

# Steps
1. Data loading and preprocessing
2. QLoRA configuration with quantization
3. Model fine-tuning with QLoRA
4. Model evaluation and saving
5. Load and test the model

In [0]:
# Create Databricks widgets to make the notebook configurable
dbutils.widgets.text("experiment_name", "fine_tuning_qlora_model")
dbutils.widgets.text("base_model", "google-bert/bert-base-uncased")
dbutils.widgets.text("run_name", "qlora-bert-base-uncased")

In [0]:
# Read the values provided through the Databricks widgets
experiment_name = dbutils.widgets.get("experiment_name")
base_model      = dbutils.widgets.get("base_model")
run_name        = dbutils.widgets.get("run_name")

## Step 0: Setup Environment

In [0]:
import json

# Import Hugging Face and PyTorch core components
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding, pipeline, EarlyStoppingCallback, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, PeftModel, prepare_model_for_kbit_training
from pyspark.sql import functions as F
import datasets
import os
import torch
import numpy as np
import evaluate

# Import MLflow for experiment tracking and model logging
import mlflow
mlflow.set_registry_uri("databricks-uc")
mlflow.set_tracking_uri("databricks")

# Logging
import warnings
warnings.filterwarnings("ignore")
import transformers
transformers.logging.set_verbosity_error()
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# GPU/CPU Setup
USE_CUDA = torch.cuda.is_available()
DEVICE = "cuda" if USE_CUDA else "cpu"

print(f"🔥 Device in uso: {DEVICE}")
if USE_CUDA:
    print(f"GPU rilevata: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Nessuna GPU rilevata — training su CPU (QLoRA richiede GPU)")

# Disable parallel tokenizers warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "0" if USE_CUDA else ""

print("PyTorch version:", torch.__version__)
print("Transformers version:", transformers.__version__)

# Check precision support
USE_BF16 = torch.cuda.is_bf16_supported() if USE_CUDA else False
USE_FP16 = USE_CUDA and not USE_BF16

In [0]:
experiment_name = f'/{experiment_name}'

train_cache_dir   = "/Volumes/main/fine_tuning_transformer_model/tmp/train"
val_cache_dir     = "/Volumes/main/fine_tuning_transformer_model/tmp/val"

model_output_dir    = "/Volumes/main/fine_tuning_transformer_model/tmp/output_model_qlora"
model_artifact_path = "classification_qlora"
training_output_dir = "/Volumes/main/fine_tuning_transformer_model/tmp/trainer_qlora"
pipeline_output_dir = "/Volumes/main/fine_tuning_transformer_model/tmp/pipeline_qlora"

In [0]:
try:
    experiment = mlflow.get_experiment_by_name(experiment_name)
    
    if experiment is None:
        experiment_id = mlflow.create_experiment(
            name=experiment_name,
            tags={'exp_name': experiment_name}
        )
        mlflow.set_experiment(experiment_id=experiment_id)
        print(f"Experiment {experiment_name} created.")
    else:
        mlflow.set_experiment(experiment_id=experiment.experiment_id)
        print(f"Experiment {experiment_name} already exists.")
except Exception as e:
    print(f"An error occurred: {e}")

## Step 1: Read Dataset

In [0]:
# Load delta tables into dataframe
train_df = spark.read.table("main.fine_tuning_transformer_model.train_data")
print(train_df.count())

test_df = spark.read.table("main.fine_tuning_transformer_model.test_data")
print(test_df.count())

In [0]:
labels = spark.read.table("main.fine_tuning_transformer_model.labels")
labels = labels.collect()

id2label = {index: row.label for (index, row) in enumerate(labels)}
label2id = {row.label: index for (index, row) in enumerate(labels)}

In [0]:
# Convert Spark DataFrame to Hugging Face Dataset
train_dataset = datasets.Dataset.from_spark(train_df, cache_dir=train_cache_dir)
test_dataset  = datasets.Dataset.from_spark(test_df, cache_dir=val_cache_dir)

## Step 2: QLoRA Configuration and Training

In [0]:
# Load Tokenizer based on the Model Name
tokenizer = AutoTokenizer.from_pretrained(base_model)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding='max_length', max_length=512, return_tensors="pt")

train_tokenized = train_dataset.map(tokenize_function, batched=True)
test_tokenized  = test_dataset.map(tokenize_function, batched=True)

### Quantization Configuration

QLoRA uses 4-bit quantization to reduce memory requirements:
- **load_in_4bit**: Enable 4-bit quantization
- **bnb_4bit_quant_type**: Type of quantization ("nf4" = NormalFloat4)
- **bnb_4bit_compute_dtype**: Computation precision (bfloat16 or float16)
- **bnb_4bit_use_double_quant**: Enable nested quantization for further memory savings

This configuration can reduce model memory footprint by ~75% compared to full precision.

In [0]:
# Configure 4-bit quantization for QLoRA
# Note: Quantization requires a CUDA-capable GPU
if USE_CUDA:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",  # NormalFloat4 quantization
        bnb_4bit_compute_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
        bnb_4bit_use_double_quant=True,  # Nested quantization for additional memory savings
    )
else:
    bnb_config = None
    print("⚠️ QLoRA requires a GPU. Proceeding without quantization.")

In [0]:
# Load the base model with 4-bit quantization
if USE_CUDA and bnb_config is not None:
    model = AutoModelForSequenceClassification.from_pretrained(
        base_model,
        num_labels=len(label2id),
        label2id=label2id,
        id2label=id2label,
        quantization_config=bnb_config,
        device_map="auto"
    )
    # Prepare model for k-bit training
    model = prepare_model_for_kbit_training(model)
else:
    # Fallback to non-quantized model if GPU is not available
    model = AutoModelForSequenceClassification.from_pretrained(
        base_model,
        num_labels=len(label2id),
        label2id=label2id,
        id2label=id2label
    )

### QLoRA Configuration

QLoRA combines LoRA with quantization:
- Uses same LoRA parameters as standard LoRA
- Applied to a 4-bit quantized base model
- Enables fine-tuning of much larger models on consumer hardware

Key difference from LoRA: The base model is quantized to 4-bit, dramatically reducing memory requirements.

In [0]:
# Configure LoRA parameters for QLoRA
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,  # Rank of low-rank matrices
    lora_alpha=32,  # Scaling factor
    lora_dropout=0.1,  # Dropout probability
    target_modules=["query", "value"],  # Apply LoRA to attention Q and V matrices
    bias="none",
    inference_mode=False
)

# Apply LoRA to the quantized model
model = get_peft_model(model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

In [0]:
training_args = TrainingArguments(
    output_dir=training_output_dir,
    per_device_train_batch_size=64 if USE_CUDA else 32,
    per_device_eval_batch_size=64 if USE_CUDA else 32,
    do_eval=True,
    do_train=True,
    num_train_epochs=3,
    learning_rate=3e-4,  # Higher learning rate for QLoRA
    optim="paged_adamw_32bit" if USE_CUDA else "adamw_torch",  # Use paged optimizer for QLoRA
    fp16=USE_FP16,
    bf16=USE_BF16,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="constant",
    dataloader_num_workers=8 if USE_CUDA else 0,
    dataloader_pin_memory=USE_CUDA,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="eval_accuracy",
    greater_is_better=True,
    logging_steps=2,
    report_to="mlflow",
    no_cuda=not USE_CUDA
)

In [0]:
data_collator = DataCollatorWithPadding(tokenizer)

In [0]:
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, predictions)}

In [0]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [0]:
try:
    with mlflow.start_run(experiment_id=experiment.experiment_id, run_name=f"trainer_{run_name}") as run:
        # Train the model with QLoRA
        trainer.train()
        
        # Save the QLoRA adapter weights and model
        trainer.model.save_pretrained(model_output_dir)
        tokenizer.save_pretrained(model_output_dir)
        
        # Merge QLoRA weights with base model for inference
        # Note: This requires dequantization and may use more memory
        merged_model = trainer.model.merge_and_unload()
        
        # Create pipeline with merged model
        pipe = pipeline(
            "text-classification",
            model=merged_model,
            tokenizer=tokenizer,
            batch_size=1
        )
        
        pipe.save_pretrained(pipeline_output_dir)
        
        # Log model to MLflow
        model_info = mlflow.transformers.log_model(
            transformers_model=pipe,
            artifact_path=model_artifact_path,
            input_example="Hi there!"
        )
        
        mlflow.end_run()

except Exception as e:
    print(f"An error occurred: {e}")
    mlflow.end_run()

## Step 3: Load and Test the Model

In [0]:
logged_model = f"runs:/{run.info.run_id}/{model_artifact_path}"
model = mlflow.pyfunc.spark_udf(spark, model_uri=logged_model, result_type='string')

In [0]:
test = test_df.select(test_df.text, test_df.label, model(test_df.text).alias("prediction"))
display(test)

In [0]:
test.write.mode("overwrite").saveAsTable("main.fine_tuning_transformer_model.prediction_qlora")

## Step 4: Register Model

In [0]:
mv = mlflow.register_model(logged_model, "main.fine_tuning_transformer_model.classification_model_qlora")
print(f"Name: {mv.name}")
print(f"Version: {mv.version}")